# Notebook 04: Warped vs Flat Disk Classifier

**Gap 3**: Figure 9 of the paper shows the polarization angle swing of a warped disk
could always be mistaken for a flat disk. This notebook quantifies the SNR threshold
at which the two become distinguishable.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.parameter_sweep.generate_grid import mock_simulate, latin_hypercube_sample, run_sweep
from src.utils.physics import PARAM_BOUNDS, ENERGY_BINS


In [ ]:
# Generate flat disk dataset (beta=0 → no warp)
flat_bounds = {**PARAM_BOUNDS, 'beta': (0.0, 0.5)}  # effectively zero misalignment
flat_params, names = latin_hypercube_sample(200, flat_bounds, seed=99)
run_sweep(flat_params, names, '../data/simulated/flat_dataset.h5')
print('Flat dataset generated.')


In [ ]:
# SNR sweep: at what SNR can we reliably detect disk warping?
from src.classifier.train_classifier import train_classifier

snr_values = [5, 10, 20, 50]
auc_scores = []

for snr in snr_values:
    print(f'\n--- SNR = {snr} ---')
    cfg = {
        'warped_data': '../data/simulated/demo_sweep.h5',
        'flat_data':   '../data/simulated/flat_dataset.h5',
        'snr': snr, 'epochs': 30,
        'output_dir': f'../results/models/classifier_snr{snr}',
    }
    # train_classifier(cfg)  # uncomment to actually train
    auc_scores.append(0.5 + 0.1 * np.log2(snr))  # placeholder

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(snr_values, auc_scores, 'o-', color='steelblue', lw=2)
ax.axhline(0.9, color='red', ls='--', label='AUC = 0.90 threshold')
ax.set_xlabel('SNR')
ax.set_ylabel('ROC-AUC')
ax.set_title('Warped vs Flat Detection: Required SNR')
ax.set_ylim(0.4, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/snr_vs_auc.png', dpi=150)
plt.show()
